In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/synthetic_loan_book.csv', parse_dates=['origination_date'])
df.shape

(10000, 14)

In [2]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   loan_id               10000 non-null  int64         
 1   loan_type             10000 non-null  str           
 2   region                10000 non-null  str           
 3   loan_purpose          10000 non-null  str           
 4   origination_date      10000 non-null  datetime64[us]
 5   orig_credit_score     10000 non-null  int64         
 6   current_credit_score  10000 non-null  int64         
 7   original_balance      10000 non-null  float64       
 8   current_balance       10000 non-null  float64       
 9   credit_limit          3039 non-null   float64       
 10  interest_rate         10000 non-null  float64       
 11  days_past_due         10000 non-null  int64         
 12  watchlist_flag        10000 non-null  int64         
 13  default_flag          10000 

loan_id                    0
loan_type                  0
region                     0
loan_purpose               0
origination_date           0
orig_credit_score          0
current_credit_score       0
original_balance           0
current_balance            0
credit_limit            6961
interest_rate              0
days_past_due              0
watchlist_flag             0
default_flag               0
dtype: int64

In [3]:
df['loan_id'].duplicated().sum()

df[df['current_balance'] > df['original_balance']].shape[0]

df[df['current_credit_score'].between(300, 850) == False].shape[0]

df[df['days_past_due'] < 0].shape[0]

0

In [4]:
df['loan_type'] = df['loan_type'].str.lower().str.strip()
df['region'] = df['region'].str.title().str.strip()
df['loan_purpose'] = df['loan_purpose'].str.lower().str.strip()

In [5]:
df['months_on_book'] = ((pd.Timestamp('2025-01-01') - df['origination_date']).dt.days / 30).round(1)

df['utilization'] = np.where(
    df['loan_type'] == 'revolving',
    (df['current_balance'] / df['credit_limit']).round(3),
    np.nan
)

df['credit_score_change'] = df['current_credit_score'] - df['orig_credit_score']

In [6]:
df.to_csv('../data/processed/clean_loan_data.csv', index=False)
print(df.shape)
df.head()

(10000, 17)


,loan_id,loan_type,region,loan_purpose,origination_date,orig_credit_score,current_credit_score,original_balance,current_balance,credit_limit,interest_rate,days_past_due,watchlist_flag,default_flag,months_on_book,utilization,credit_score_change
0,1,term,East,personal,2024-12-25,633,617,7835.88,7697.92,NaN,16.96,9,0,0,0.2,NaN,-16
1,2,revolving,East,personal,2024-10-05,727,713,18202.97,17050.19,36929.54,9.82,1,0,0,2.9,0.462,-14
2,3,term,South,auto,2023-08-02,625,600,19197.03,14505.45,NaN,9.65,5,0,0,17.3,NaN,-25
3,4,term,South,auto,2022-12-19,613,565,8637.39,4821.45,NaN,7.25,14,0,0,24.8,NaN,-48
4,5,term,West,personal,2022-08-17,752,732,19720.10,3186.66,NaN,7.02,2,1,0,28.9,NaN,-20


# Data Cleaning

Loads the raw synthetic loan book, validates data integrity (no duplicate IDs, no impossible values),
standardizes categorical fields, and engineers derived fields needed for later staging and modeling:
months_on_book, utilization (revolving only), and credit_score_change (origination vs current — 
a direct input to SICR detection in the staging engine).

Note: credit_limit and utilization are null for term loans by design — term loans do not have a
revolving credit limit, so this is not missing data.